# Bab 01: Categorical Encoding Fundamentals

## Fokus Bab

Bab ini membangun landasan konseptual sebelum teknik encoding diimplementasikan secara mendalam pada Bab 02 dan 03. Fokusnya adalah memahami karakteristik variabel kategorik (jenis, jumlah kategori, dan cara model memperlakukannya), sehingga pemilihan metode encoding dapat dilakukan secara beralasan, bukan berdasarkan kebiasaan.

## Tujuan Pembelajaran

1. Membedakan fitur nominal, ordinal, dan high-cardinality.
2. Memahami bagaimana kategori direpresentasikan agar dapat diproses model Machine Learning.
3. Menentukan strategi encoding berdasarkan karakteristik data dan algoritma.

In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# 1. Fundamental Variabel Kategorik

## 1.1 Pengertian Variabel Kategorik

* Variabel kategorik adalah variabel yang nilainya berasal dari himpunan terbatas label atau kelompok, bukan dari hasil pengukuran kuantitatif.
* Setiap nilai menyatakan keanggotaan pada suatu kelompok misalnya kota, metode pembayaran, jenis produk, bukan menunjukan besaran.
* Operasi aritmatika pada kategori tidak bermakna

Pada pandas, kolom kategorik dapat tersimpan dalam beberapa bentuk:


| Representasi | Keterangan |
|---|---|
| `object` / `str` | Teks biasa. Pada pandas 3.0, kolom teks berdtype `str`; pada versi sebelumnya `object`. |
| `category` | Tipe khusus pandas: kategori disimpan satu kali, data disimpan sebagai kode bilangan bulat. |
| `bool` | Kasus khusus dengan dua kategori. |

Implementasi

In [14]:
df = pd.DataFrame({
    "kota":         ["Jakarta", "Bandung", "Surabaya", "Jakarta", "Medan", "Bandung"],
    "metode_bayar": ["QRIS", "Transfer", "COD", "QRIS", "Transfer", "QRIS"],
    "ukuran":       ["S", "M", "L", "XL", "M", "S"],
    "jumlah_item":  [2, 1, 3, 5, 1, 2],
})

print(df.dtypes)

kota_cat = df["kota"].astype("category")

print(kota_cat.cat.categories)
print(kota_cat.cat.codes.tolist())

kota              str
metode_bayar      str
ukuran            str
jumlah_item     int64
dtype: object
Index(['Bandung', 'Jakarta', 'Medan', 'Surabaya'], dtype='str')
[1, 0, 3, 1, 2, 0]


- **`df.dtypes`** — memeriksa tipe data setiap kolom pada DataFrame.
- **`astype("category")`** — mengubah kolom teks menjadi tipe data kategorik.
- **`.cat.categories`** — menampilkan daftar kategori unik pada kolom kategorik.
- **`.cat.codes`** — menampilkan kode bilangan bulat internal yang merepresentasikan setiap kategori.

Kasus di Dunia Nyata

Sebuah platform e-commerce menyimpan kolom kota, metode_bayar, dan kategori_produk pada tabel transaksi. Ketiganya berupa teks dan tidak dapat langsung dipakai oleh sebagian besar algoritma Machine Learning, padahal kolom tersebut sering menjadi prediktor penting (misalnya untuk memprediksi pembatalan pesanan).

Interpretasi 

Kode [1, 0, 3, 1, 2, 0] dihasilkan secara alfabetis (Bandung=0, Jakarta=1, Medan=2, Surabaya=3), bukan berdasarkan makna apa pun. Artinya, kode ini hanyalah label internal. Jangan menyimpulkan bahwa Surabaya "lebih besar" daripada Jakarta. Pemahaman ini menjadi dasar pembahasan risiko urutan numerik palsu pada Subbab 2.3

## 1.2 Nominal vs Ordinal

### Dasar Pembedaan

* Variabel kategorik tidak selalu setara, Dua jenis utama dibedakan ada atau tidaknya urutan bermakna di antara kategori.
* Pertanyaan kuncinya: Apakah kategori hanya bisa dibedakan, atau juga bisa diurutkan?



![0900](../assets/figures/01_gambar1_taksonomi_kategorik.png)

### Nominal

* Kategori tidak memiliki urutan alami
* Yang bermakna hanyalah kesamaan atau perbedaan label, bukan "lebih besar/kecil" atau "lebih tinggi atau rendah"
* Contoh:
    * Kota: jakarta, surabaya, bandung.
    * Metode pembayaran: QRIS, Tunai, e-wallet.
* Operasi masuk akal:
    * menghitung frekuensi tiap kategori.
    * Menghitung modus.
    * Membuat tabel kontingensi.
    * one-hot encoding
* Operasi tidak bermakna: Aritmatika.
* Jika dikodekan sebagai angka, angka tersebut hanya identifier, bukan besaran. Contoh: 1 = Jakarta, 2 = Bandung tidak berarti Bandung “dua kali” Jakarta.

### Ordinal

* Kategori memiliki urutan bermakna.
* Contoh:
    * S < M < L
    * SD < SMP < SMA
* Operasi masuk akal
    * Membandingkan urutan > misalnya “puas” lebih tinggi daripada “netral”,
    * Mencari minimum dan maksimum kategori.
    * Median jika urutan dan definisi jelas.
    * Modus.
* Operasi yang belum tentu bermakna:
    * rata-rata,
    * selisih antarkategori,
    * rasio antarkategori.
* Alasan: urutan diketahui, tetapi jarak antartingkat belum tentu sama.
    * Selisih “puas” dan “sangat puas” tidak harus identik dengan selisih “netral” dan “puas”.
    * Karena itu, data ordinal tidak otomatis boleh diperlakukan seperti data numerik interval.

### Implikasi penting

* Nominal    
    * Jangan memberi kode integer lalu memperlakukannya sebagai numerik dalam model linear.
    * Gunakan one-hot encoding atau representasi yang tidak menyiratkan urutan.
    * Visualisasi sebaiknya tidak memaksa urutan tertentu kecuali untuk kerapian tampilan.

* Ordinal
    * Urutan kategori harus didefinisikan secara eksplisit.
    * Dalam pandas, gunakan category dengan ordered=True dan tentukan urutan lewat categories.
    * Encoding ordinal boleh menggunakan angka yang mengikuti urutan, tetapi tetap hati-hati: angka itu menunjukkan peringkat, bukan jarak yang setara.

* Kesalahan umum
    * Menganggap angka pada label selalu numerik. Contoh: kode pos 40123 bukan variabel kuantitatif.
    * Menganggap ordinal boleh langsung dirata-ratakan. Rata-rata baru bermakna jika skala jaraknya dapat dijustifikasi.
    * Menganggap nominal bisa diurutkan hanya karena labelnya berupa angka.

### Implementasi

Pandas menyediakan CategoricalDtype dengan parameter ordered=True untuk menyatakan urutan secara eksplisit.

In [15]:
ukuran_dtype = pd.CategoricalDtype(["S", "M", "L", "XL"], ordered=True)
df['ukuran'] = df['ukuran'].astype(ukuran_dtype)


print(df["ukuran"].dtype)
print(df["ukuran"].cat.codes.tolist())
print((df["ukuran"] >= "L").tolist())

category
[0, 1, 2, 3, 1, 0]
[False, False, True, True, False, False]


Penjelasan kode:
> Urutan kategori ditetapkan manual melalui daftar ["S", "M", "L", "XL"]. Setelah bertipe ordered, pandas mengizinkan perbandingan seperti >= "L". Tanpa ordered=True, perbandingan tersebut akan menimbulkan galat.

Kasus di Dunia Nyata:
> Toko pakaian daring ingin memprediksi kemungkinan retur. Kolom ukuran bersifat ordinal, sedangkan metode_bayar bersifat nominal. Memperlakukan keduanya dengan cara yang sama akan menghilangkan informasi urutan pada ukuran atau menciptakan urutan palsu pada metode_bayar.

Interpretasi:
> Hasil [False, False, True, True, False, False] menunjukkan bahwa pandas memahami L dan XL sebagai ukuran yang tidak lebih kecil dari L. Urutan alfabetis (L < M < S < XL) akan keliru, sehingga urutan harus selalu dinyatakan secara eksplisit.

## 1.3 Cardinality: Low vs High

### Konsep

* Cardinality: Jumlah kategori unik pada suatu kolom.
* Cardinality mempengaruhi secara langsung pemilihan encoding.
* One hot encoding menambahkan satu kolom untuk setiap kategori.

![Cardinality](../assets/figures/01_gambar2_cardinality.png)

* Low: kurang dari kira kira 10 kategori.
* Menengah: puluhan hingga ratusan
* High: ribuan kategori atau lebih

### Implementasi

In [16]:
rng = np.random.default_rng(42)
n = 5000
big = pd.DataFrame({
    "status_pesanan": rng.choice(["baru", "dibayar", "dikirim", "selesai", "batal"], n),
    "metode_bayar":   rng.choice(["QRIS", "Transfer", "COD", "Kartu", "E-wallet", "Paylater"], n),
    "kota":           rng.choice([f"kota_{i}" for i in range(300)], n),
    "id_produk":      rng.choice([f"SKU{i:05d}" for i in range(4000)], n),
    "id_pelanggan":   [f"C{i:05d}" for i in rng.integers(0, 3000, n)],
})

ringkasan = pd.DataFrame({
    "n_unik": big.nunique(),
    "rasio_unik": (big.nunique() / len(big)).round(3),
}).sort_values("n_unik")
print(ringkasan)

                n_unik  rasio_unik
status_pesanan       5       0.001
metode_bayar         6       0.001
kota               300       0.060
id_pelanggan      2439       0.488
id_produk         2831       0.566


Penjelasan kode:
> nunique() menghitung jumlah kategori unik per kolom. Rasio unik (n_unik / jumlah baris) membantu menilai apakah suatu kolom lebih menyerupai pengenal (identifier) daripada atribut kategorik.

Kasus di Dunia Nyata
> Pada data marketplace, status_pesanan hanya memiliki beberapa nilai, kota memiliki ratusan nilai, sedangkan id_produk dan id_pelanggan memiliki ribuan hingga jutaan nilai. Keempatnya adalah kolom kategorik, tetapi tidak dapat diperlakukan dengan cara yang sama.

Interpretasi
> status_pesanan dan metode_bayar (5 dan 6 kategori) tergolong low-cardinality. kota (300 kategori) berada di tingkat menengah. id_produk dan id_pelanggan memiliki rasio unik sekitar 0,5, artinya rata-rata hanya sekitar dua baris per kategori, sehingga statistik per kategori sangat tidak stabil. Jika seluruh kolom ini di-one-hot-encode, jumlah kolom yang terbentuk mencapai 5 + 6 + 300 + 2.831 + 2.439 = 5.581 kolom dari hanya lima fitur asal. Inilah alasan strategi khusus diperlukan untuk high-cardinality (Subbab 3.3).

## 1.4 Representasi Kategori dalam Model

* Inti masalah: estimator scikit-learn membutuhkan x numerik; tipe kategori/string harus di encode dulu.
* Nominal (tanpa urutan): gunakan one-hot encoding. Jangan pakai integer 0,1,2 karena model linear bisa salah menganggapnya berurutan/berjarak.
* Ordinal (ada urutan):  boleh pakai ordinal encoding dengan urutan eksplisit, misalnya S=0, M=1, L=2. Ingat: kode hanya peringkat, bukan jarak yang setara.
* Konsistensi: encoding harus di-fit pada train saja, lalu transform data test.Tempatkan di Pipeline/ColumnTransformer agar tidak leakage.
* Kategori baru: set handle_unknown="ignore" (atau sejenis) agar data inference tidak error.
* Catatan: LabelEncoder untuk target y, bukan fitur X. Sebagian model tree punya dukungan kategorik native, tetapi harus dicek dokumentasinya. 



| Pendekatan | Cara kerja | Dibahas pada |
|---|---|---|
| Kode bilangan bulat | Setiap kategori diberi satu angka | Bab 02 |
| Vektor indikator (One-Hot) | Satu kolom biner per kategori | Bab 02 |
| Statistik kategori | Frekuensi atau rata-rata target per kategori | Bab 03 |
| Vektor terpelajar (Embedding) | Vektor padat yang dilatih bersama model | Bab 03 |

Implementasi

In [17]:
# Representasi indikator
print(pd.get_dummies(df[["kota"]], dtype=int))

# Representasi dengan encoder scikit-learn
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X = ohe.fit_transform(df[["kota", "metode_bayar"]])
print(ohe.get_feature_names_out())
print(X.shape)

   kota_Bandung  kota_Jakarta  kota_Medan  kota_Surabaya
0             0             1           0              0
1             1             0           0              0
2             0             0           0              1
3             0             1           0              0
4             0             0           1              0
5             1             0           0              0
['kota_Bandung' 'kota_Jakarta' 'kota_Medan' 'kota_Surabaya'
 'metode_bayar_COD' 'metode_bayar_QRIS' 'metode_bayar_Transfer']
(6, 7)


Penjelasan Kode: One-Hot Encoding

Bagian 1 — `pd.get_dummies`
```python
print(pd.get_dummies(df[["kota"]], dtype=int))
```
- `pd.get_dummies` mengubah tiap kategori menjadi kolom biner 0/1, dengan pola nama seperti `kota_Jakarta`, `kota_Bandung`.
- `df[["kota"]]` memilih kolom sebagai DataFrame, bukan Series.
- `dtype=int` membuat hasil berupa `0`/`1`; default-nya `bool`.
- Cocok untuk eksplorasi cepat, tetapi kurang aman untuk pipeline karena kolom bisa berbeda antara data train dan test.

Bagian 2 — `OneHotEncoder`
```python
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X = ohe.fit_transform(df[["kota", "metode_bayar"]])
print(ohe.get_feature_names_out())
print(X.shape)
```
- `OneHotEncoder(...)` adalah encoder scikit-learn yang bisa disimpan dan dipakai ulang; lebih cocok untuk pipeline.
- `handle_unknown="ignore"` membuat kategori baru saat `transform` tidak error, tetapi menghasilkan semua nol. Alternatifnya `"error"` (default) dan `"infrequent_if_exist"`.
- `sparse_output=False` menghasilkan array NumPy dense; `True` (default) menghasilkan sparse matrix.
- `fit_transform(...)` mempelajari kategori lalu mengubah data; hanya untuk data train. Untuk data test gunakan `ohe.transform(...)` agar tidak leakage.
- `get_feature_names_out()` menampilkan nama kolom hasil encoding, misalnya `['kota_Bandung', 'kota_Jakarta', 'metode_bayar_kartu', ...]`.
- `X.shape` menunjukkan ukuran array `(jumlah_baris, jumlah_kolom)`.

Ringkasan

| Aspek | `pd.get_dummies` | `OneHotEncoder` |
|---|---|---|
| Asal | pandas | scikit-learn |
| Bisa dipakai ulang | Tidak praktis | Ya, lewat `fit`/`transform` |
| Konsistensi train–test | Rawan | Terjamin |
| Kategori baru | Menambah kolom | Ditangani `handle_unknown` |
| Cocok untuk pipeline | Kurang | Ya |

Intinya: `get_dummies` untuk eksplorasi cepat; `OneHotEncoder` untuk pipeline produksi karena konsisten dan aman terhadap kategori tak dikenal.

Kasus di Dunia Nyata:
> Sistem rekomendasi toko daring mengubah kolom kota dan metode_bayar menjadi 7 kolom biner sebelum dimasukkan ke model regresi logistik untuk memprediksi pembatalan pesanan.

Interpretasi
> Dua kolom asal berubah menjadi tujuh kolom karena terdapat 4 kota dan 3 metode pembayaran. Tiap baris hanya memiliki satu nilai 1 per kolom asal. Pada bab-bab berikutnya akan terlihat bahwa pilihan representasi memengaruhi jumlah fitur, kebutuhan memori, dan perilaku model.

# 2. Mengapa Encoding Diperlukan

## 2.1 Kategori sebagai Teks

* Algoritma machine learning melalui operasi matemtis seperti pekalian matrix, perhitungan jarak, dan turunan. . Operasi-operasi ini tidak terdefinisi pada string, sehingga teks kategorik tidak bisa langsung dipakai sebagai masukan model.
* Teks kategorik harus diubah kedalam numerik sebelum masuk ke model.Namun encoding hanya boleh dilakukan setelah teksnya konsisten.
* Teks kategorik rentan terhadap inkonsistensi penulisan. Satu kategori yang sama bisa terpecah menjadi beberapa bentuk, misalnya Jakarta, jakarta, JKT, Jakarta, atau salah ketik.
* Jika dibiarkan, inkonsistensi ini membuat frekuensi kategori salah, hasil encoding tidak stabil, dan model dapat mempelajari pola yang sebenarnya tidak ada.
* Langkah yang perlu dilakukan sebelum encoding:
    1. normalisasi huruf, misalnya menyamakan huruf besar dan kecil;
    2. menghapus spasi berlebih di awal dan akhir;
    3. menyeragamkan singkatan dan alias, misalnya JKT menjadi Jakarta;
    4. memperbaiki salah ketik yang jelas;
    5. memvalidasi nilai terhadap daftar kategori yang diizinkan.
* Setelah teks kategorik konsisten, barulah dilakukan encoding, misalnya one-hot encoding untuk kategori nominal atau ordinal encoding untuk kategori ordinal.

> Intinya: string tidak bisa langsung dioperasikan secara matematis, dan teks kategorik harus dibersihkan serta diseragamkan sebelum diubah menjadi angka.



In [18]:
from sklearn.linear_model import LogisticRegression

data = pd.DataFrame({
    "kota": ["Jakarta", "Bandung", "Surabaya", "Jakarta", "Medan", "Bandung"],
    "batal": [1, 0, 1, 1, 0, 0],
})

try:
    LogisticRegression().fit(data[["kota"]], data[["batal"]])
except ValueError as e:
    print("Value Error", e)

# Inkonsistensi penulisan
mentah = pd.Series(["Jakarta", " jakarta", "JAKARTA ", "Bandung", "bandung "])

print(mentah.nunique())
print(mentah.str.strip().str.lower().nunique())

Value Error could not convert string to float: 'Jakarta'
5
2


- Model menolak masukan berupa teks karena tidak dapat mengubah nilai seperti `'Jakarta'` menjadi bilangan yang bisa diproses secara matematis.
- Pada contoh kedua, `str.strip()` menghapus spasi berlebih di awal dan akhir teks.
- `str.lower()` mengubah semua huruf menjadi huruf kecil agar penulisan seragam.
- Keduanya digabung menjadi `str.strip().str.lower()` untuk menyeragamkan spasi dan kapitalisasi sebelum teks di-encoding.

Kasus di Dunia Nyata:
> Data pendaftaran pelanggan yang diisi manual sering memuat “Jakarta”, “jakarta”, dan “JAKARTA “ sebagai entri berbeda. Tanpa pembersihan, satu kota dianggap sebagai tiga kategori terpisah.

Interpretasi
> Pesan galat menegaskan bahwa encoding bersifat wajib, bukan opsional. Penurunan dari 5 menjadi 2 kategori menunjukkan bahwa pembersihan teks harus dilakukan sebelum encoding; jika tidak, encoder akan mempertahankan duplikasi dan menambah dimensi tanpa alasan.

## 2.2 Kategori sebagai Angka

- Tipe data numerik di pandas atau database hanya menunjukkan cara penyimpanan, bukan makna statistik. Bilangan bulat bisa berperan sebagai angka, label, atau sekadar identifier.
- Contoh kolom yang sering tampil sebagai angka tetapi sebenarnya kategorik:
  - kode pos, misalnya `40123`;
  - kode wilayah administratif;
  - ID pelanggan, ID transaksi, NIK;
  - kode status, misalnya `1 = aktif`, `2 = nonaktif`;
  - nomor kamar, nomor polisi, atau kode produk.
- Uji praktis: apakah operasi aritmatika pada nilai tersebut bermakna?
  - Jika rata-rata, selisih, penjumlahan, atau rasio tidak punya arti, kolom itu kategorik.
  - Contoh: rata-rata kode pos `40123` dan `40234` tidak bermakna.
  - Contoh: selisih kode status `2 - 1 = 1` tidak berarti “selisih satu tingkat” kecuali status memang ordinal dan urutannya didefinisikan.
- Uji tambahan yang membantu:
  - Apakah angka hanya berfungsi membedakan objek? Jika ya, itu identifier, bukan besaran.
  - Apakah nilai memiliki nol yang bermakna? Kode pos `01234` yang hilang nol depan menunjukkan angka diperlakukan sebagai teks/label.
  - Apakah jarak antarnilai bermakna? Jika tidak, jangan diperlakukan sebagai numerik interval.
  - Apakah urutan bermakna? Jika tidak, jangan ordinal encoding; gunakan one-hot.
- Implikasi pemodelan:
  - Jangan masukkan ID sebagai fitur numerik karena model bisa mempelajari pola acak dan menyebabkan overfitting atau leakage.
  - Untuk kategori nominal seperti kode pos, gunakan one-hot encoding atau teknik high-cardinality yang sesuai.
  - Untuk kode status yang ordinal, gunakan ordinal encoding dengan urutan eksplisit.
  - Untuk ID unik, biasanya kolom dihapus atau diperlakukan khusus, bukan dijadikan fitur biasa.
- Kesalahan umum:
  - Menganggap `dtype` numerik otomatis berarti variabel kuantitatif.
  - Menghitung rata-rata kode pos, kode wilayah, atau nomor ID.
  - Memberi kode integer pada kategori nominal lalu memperlakukannya sebagai angka berurutan.
- Intinya: pertanyaan kuncinya bukan “apakah kolom berisi angka?”, tetapi “apakah operasi aritmatika pada angka itu bermakna?”. Jika tidak, kolom tersebut kategorik.

Implementasi

In [19]:
pesanan = pd.DataFrame({
    "kode_pos": [40111, 10110, 60234, 40111],
    "kode_status": [1, 2, 3, 1],        # 1=baru, 2=dikirim, 3=batal
    "jumlah_item": [2, 1, 3, 5],         # kuantitas sejati
})

# Perlakukan kolom bermakna kategori sebagai kategori
pesanan["kode_pos"] = pesanan["kode_pos"].astype("category")
pesanan["kode_status"] = pesanan["kode_status"].astype("category")
print(pesanan.dtypes)

kode_pos       category
kode_status    category
jumlah_item       int64
dtype: object


- Kolom yang secara makna adalah kategori, misalnya kode pos, kode wilayah, ID, atau kode status, dikonversi secara eksplisit menjadi tipe `category`.
- Konversi eksplisit ini menandai bahwa nilai tersebut adalah label atau identifier, bukan besaran yang boleh dijumlahkan, dikurangkan, atau dirata-ratakan.
- Contoh konversi:
  ```python
  df["kode_pos"] = df["kode_pos"].astype("category")
  df["kode_status"] = df["kode_status"].astype("category")
  ```
- Kolom `jumlah_item` tetap dibiarkan numerik karena nilainya merupakan kuantitas sebenarnya, sehingga operasi aritmatika seperti total, rata-rata, atau selisih tetap bermakna.
- Pengecekan tipe setelah konversi bisa dilakukan dengan `df.dtypes` untuk memastikan kolom kategori tidak lagi diperlakukan sebagai angka.
- Konversi ke `category` belum tentu langsung siap untuk model; tahap encoding seperti one-hot atau ordinal encoding tetap dilakukan sesuai jenis kategorinya.
- Intinya: kolom yang bermakna kategori ditandai eksplisit sebagai kategori agar tidak diperlakukan sebagai besaran, sedangkan kolom kuantitas sebenarnya seperti `jumlah_item` tetap numerik.

Kasus di Dunia Nyata
> Pada data logistik, kode_pos berupa angka lima digit. Jika dimasukkan apa adanya ke regresi linear, model mengasumsikan bahwa kode pos 60234 “tiga kali lipat lebih besar” pengaruhnya daripada 20078, sebuah asumsi yang tidak memiliki dasar.

Interpretasi
> Tipe data pada tabel hanyalah cerminan cara penyimpanan, bukan makna statistik. Peneliti perlu memeriksa makna kolom (melalui kamus data atau pemangku kepentingan bisnis), bukan sekadar tipenya.

## 2.3 Risiko Urutan Numerik Palsu

- Kategori nominal tidak punya urutan maupun jarak. Jika diberi kode bilangan bulat seperti `Jakarta=0`, `Bandung=1`, `Surabaya=2`, angka itu hanya identifier, bukan besaran.
- Model tidak tahu bahwa kode tersebut hanyalah label. Ia akan memperlakukan angka sebagai nilai numerik yang berurutan dan berjarak.
- Akibatnya muncul dua informasi palsu:
  - urutan palsu, seolah `Surabaya > Bandung > Jakarta`;
  - jarak palsu, seolah selisih `Jakarta` ke `Bandung` sama dengan `Bandung` ke `Surabaya`.
- Pada model linear, misalnya regresi atau regresi logistik, satu koefisien harus menggambarkan pengaruh yang berubah secara konstan setiap kode naik satu.
  - Model dipaksa mengasumsikan hubungan linier dan monoton terhadap kode.
  - Padahal pengaruh kategori nominal bisa tidak berurutan, tidak linier, atau berbeda-beda tanpa pola jarak.
- Pada model berbasis jarak, misalnya KNN, K-Means, atau SVM dengan kernel jarak, kategori dengan kode berdekatan dianggap lebih mirip.
  - `Jakarta=0` dan `Bandung=1` dianggap lebih mirip daripada `Jakarta=0` dan `Surabaya=2`.
  - Padahal kedekatan angka itu tidak punya makna.
- Visualisasi pada `01_gambar3_urutan_numerik_palsu.png` memperlihatkan perbedaan antara kategori yang sebenarnya tidak berurutan dan representasi angka yang seolah membuatnya berurutan.
- Dampak lanjutan:
  - model dapat mempelajari pola yang tidak ada pada data asli;
  - prediksi menjadi bias dan sulit diinterpretasi;
  - hasil bisa berubah hanya karena urutan pemberian kode, bukan karena perbedaan kategori yang sebenarnya.
- Solusi:
  - untuk kategori nominal, gunakan one-hot encoding agar tidak menyiratkan urutan atau jarak;
  - untuk kategori ordinal, ordinal encoding boleh dipakai, tetapi urutan harus didefinisikan eksplisit dan jarak antarkategori tetap tidak dianggap setara;
  - jangan memberi kode integer pada nominal lalu memperlakukannya sebagai fitur numerik biasa.
- Intinya: label encoding pada kategori nominal menciptakan urutan dan jarak palsu. Model linear memaksakan pengaruh konstan per kenaikan kode, sedangkan model berbasis jarak menganggap kode berdekatan lebih mirip. Keduanya berisiko menghasilkan model yang salah secara makna.

![01_gambar3_urutan_numerik_palsu](../assets/figures/01_gambar3_urutan_numerik_palsu.png)

Implementasi

In [20]:
# Label encoding pada nominal: Jakarta=0, Bandung=1, Surabaya=2
kode = np.array([0, 1, 2])
print("Jarak Jakarta - Surabaya", abs(kode[0] - kode[2]))

# One-hot: jarak euclidean antar kategori
from itertools import combinations

I = np.eye(3)
jarak = [round(float(np.linalg.norm(I[i] - I[j])), 3)
         for i, j in combinations(range(3), 2)]
print("Jarak One-Hot:", jarak)

Jarak Jakarta - Surabaya 2
Jarak One-Hot: [1.414, 1.414, 1.414]


- `kode = np.array([0, 1, 2])` membuat array NumPy berisi kode integer untuk tiga kategori: `Jakarta=0`, `Bandung=1`, `Surabaya=2`.
- `kode[0]` mengambil kode Jakarta, `kode[2]` mengambil kode Surabaya.
- `abs(kode[0] - kode[2])` menghitung selisih absolut `0 - 2 = -2`, lalu menjadi `2`. Artinya, label encoding langsung menganggap Jakarta–Surabaya berjarak `2`.
- Jarak `2` ini tidak berasal dari data asli, melainkan hanya akibat urutan pemberian kode. Jika urutan kode diubah, jaraknya juga berubah.
- `from itertools import combinations` mengimpor fungsi untuk membuat pasangan indeks tanpa pengulangan.
- `combinations(range(3), 2)` menghasilkan pasangan `(0,1)`, `(0,2)`, dan `(1,2)`, yaitu semua pasangan kategori berbeda.
- `I = np.eye(3)` membuat matriks identitas 3×3. Setiap barisnya adalah vektor one-hot:
  - `I[0] = [1, 0, 0]` untuk Jakarta,
  - `I[1] = [0, 1, 0]` untuk Bandung,
  - `I[2] = [0, 0, 1]` untuk Surabaya.
- `I[i] - I[j]` menghitung selisih dua vektor one-hot. Untuk kategori berbeda, hasilnya berisi dua nilai `1` dan `-1`.
- `np.linalg.norm(...)` menghitung jarak Euclidean dari selisih vektor tersebut.
- Untuk setiap pasangan kategori berbeda, normanya adalah `sqrt(1^2 + 1^2) = sqrt(2) ≈ 1.414`.
- `round(float(...), 3)` membulatkan hasil menjadi tiga angka desimal agar mudah dibaca.
- Hasil `jarak` adalah `[1.414, 1.414, 1.414]`, artinya semua pasangan kategori memiliki jarak yang sama.
- Pada label encoding, jarak antarkategori berbeda-beda dan bergantung pada urutan kode. Pada one-hot encoding, semua kategori berbeda berjarak sama, yaitu `√2`.
- Maknanya: untuk kategori nominal, one-hot lebih tepat karena tidak menyiratkan urutan atau kedekatan palsu. Namun one-hot juga tidak menyatakan bahwa dua kategori “mirip”; ia hanya menyatakan bahwa keduanya berbeda.

Kasus di Dunia Nyata:
> Model prediksi ongkos kirim memakai kota_asal yang di-label-encode berdasarkan urutan alfabet. Algoritma KNN lalu menganggap “Bandung” (kode 1) lebih mirip dengan “Jakarta” (kode 0) daripada dengan “Surabaya” (kode 2), padahal pengurutan itu hanya hasil kebetulan abjad.

Interpretasi:
> Jarak Jakarta–Surabaya sebesar 2 versus Jakarta–Bandung sebesar 1 adalah artefak encoding, bukan fakta tentang kota. One-Hot menghilangkan artefak ini karena seluruh kategori berjarak sama. Perlu dicatat, model berbasis tree lebih toleran terhadap kode bilangan bulat karena hanya membuat pemisahan berdasarkan ambang, tetapi urutan kode yang sewenang-wenang tetap dapat membuat pemisahan menjadi kurang efisien (dibahas pada Subbab 3.4).

# 3. Strategi Pemilihan Encoding

Gambar 4 (01_gambar4_alur_pemilihan_encoding.png) merangkum alur penentuan strategi yang dibahas pada seluruh subbab berikut.

![01_gambar4_alur_pemilihan_encoding](../assets/figures/01_gambar4_alur_pemilihan_encoding.png)

## 3.1 Encoding untuk Nominal

- Kategori nominal dengan cardinality rendah berarti jumlah kategori uniknya sedikit. Pada kondisi ini, one-hot encoding menjadi pilihan standar karena dimensi hasil encoding masih wajar dan mudah dikelola.
- One-hot encoding membuat satu kolom biner per kategori. Setiap baris bernilai `1` pada kategori miliknya dan `0` pada kategori lain, sehingga tidak menyiratkan urutan atau jarak antar kategori.
- Dokumentasi scikit-learn menyebutkan encoding semacam ini diperlukan untuk memasukkan data kategorik ke banyak estimator, khususnya model linear dan SVM dengan kernel standar, karena estimator tersebut bekerja pada matriks numerik.
- Model linear dan SVM standar tidak memahami string atau tipe `category` secara native. Tanpa encoding, data kategorik tidak bisa diproses. Jika diberi label encoding, model linear justru bisa menganggap kode integer sebagai besaran berurutan.
- One-hot encoding menghindari asumsi urutan palsu tersebut. Namun konsekuensinya, dimensi fitur bertambah sebanyak jumlah kategori, sehingga kurang cocok untuk kategori dengan cardinality sangat tinggi.
- Dua parameter penting pada `OneHotEncoder`:
  - `handle_unknown` mengatur perilaku saat `transform` menemukan kategori yang tidak pernah dilihat saat `fit`.
    - `"error"` (default): memunculkan error jika ada kategori baru.
    - `"ignore"`: kategori baru menghasilkan semua nol pada kolom terkait, tanpa error.
    - `"infrequent_if_exist"`: kategori jarang dan kategori tak dikenal digabung ke kategori khusus, biasanya perlu diatur bersama `min_frequency` atau `max_categories`.
  - `drop` mengatur apakah satu kolom dibuang per fitur untuk mengurangi redundansi.
    - `None` (default): semua kolom kategori dipertahankan.
    - `"first"`: membuang kategori pertama pada setiap fitur, berguna untuk menghindari multikolinearitas sempurna atau dummy variable trap pada model linear dengan intercept.
    - `"if_binary"`: membuang satu kolom hanya untuk fitur biner.
- Pada model linear, membuang satu kolom membuat kategori yang tersisa diinterpretasikan relatif terhadap kategori acuan yang dibuang. Ini membantu interpretasi koefisien dan mengurangi redundansi.
- Pada model dengan regularisasi, multikolinearitas tidak selalu fatal, tetapi `drop="first"` tetap bisa membantu stabilitas dan interpretasi. Pada model berbasis pohon, `drop` biasanya tidak diperlukan karena model tidak terlalu terganggu redundansi linear.
- Encoding harus di-`fit` hanya pada data train, lalu `transform` diterapkan pada data validasi/test untuk mencegah leakage. Gunakan `Pipeline` dan `ColumnTransformer` agar proses ini konsisten.
- Intinya: untuk nominal cardinality rendah, one-hot encoding adalah pilihan standar karena aman secara makna dan cocok untuk model linear serta SVM. Atur `handle_unknown` agar tahan terhadap kategori baru, dan pertimbangkan `drop` untuk mengurangi redundansi pada model linear tertentu.

Implementasi

In [21]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(df[["kota", "metode_bayar"]])

# Kategori baru "Makassar" yang tidak ada saat fit
baru = pd.DataFrame({"kota": ["Makassar"], "metode_bayar": ["QRIS"]})
print(ohe.transform(baru))

[[0. 0. 0. 0. 0. 1. 0.]]


- `ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)` membuat encoder one-hot yang tidak error saat menemukan kategori baru, dan menghasilkan array dense.
- `ohe.fit(df[["kota", "metode_bayar"]])` mempelajari daftar kategori unik dari data train untuk kedua kolom. Daftar ini disimpan di `ohe.categories_` dan tidak berubah saat `transform`.
- `baru = pd.DataFrame(...)` membuat data baru berisi `kota="Makassar"` dan `metode_bayar="QRIS"`. Data ini hanya ditransform, bukan di-fit ulang.
- `ohe.transform(baru)` mengubah data baru memakai kategori yang sudah dipelajari saat fit.
- Karena `"Makassar"` tidak ada saat fit dan `handle_unknown="ignore"`, seluruh kolom hasil encoding untuk fitur `kota` diisi `0`. Tidak ada error.
- Kolom `metode_bayar_QRIS` bernilai `1` hanya jika `"QRIS"` sudah pernah muncul saat fit. Jika `"QRIS"` juga kategori baru, seluruh kolom `metode_bayar` juga akan `0`.
- Dengan `sparse_output=False`, hasil `transform` berupa array NumPy dense berisi `0` dan `1`.
- Urutan kolom mengikuti `ohe.get_feature_names_out()`, misalnya `kota_Bandung`, `kota_Jakarta`, `metode_bayar_kartu`, `metode_bayar_QRIS`, dan seterusnya.
- Kategori baru tidak ditambahkan ke `categories_`. Encoder tetap konsisten dengan kategori saat fit.
- Baris dengan kategori tak dikenal bisa menjadi semua nol pada fitur tersebut. Model dapat memperlakukannya sebagai baseline atau “tidak diketahui”, tetapi jika kategori baru sering muncul, pertimbangkan `handle_unknown="infrequent_if_exist"` atau perbarui encoder.
- Intinya: `handle_unknown="ignore"` membuat kategori baru aman diproses tanpa error, dengan konsekuensi kolom fitur terkait menjadi semua nol.